# LightGCN Top-100 — Kaggle 출력 보장 수정판 (v2)

**실행 방법:** Kaggle에서 이 `.ipynb` **하나만 Import**하고 `Add Input`으로 **대회 원본**과 **공통 Validation Dataset (`swaidataset/amazon-topk-teamkit`)**을 연결한 후 **GPU T4**를 선택하세요. **`Save Version → Save & Run All`**로 실행해야 결과가 **저장된 버전의 Output**에 보존됩니다.

- 이미 같은 세션에서 생성된 CSV가 있으면 **재학습 없이 재사용**합니다.
- 없으면 LightGCN Seed 42 / Dimension 128 / Layers 3 / LR 0.003 / Epoch 575를 **Validation / Final 두 번** 학습합니다.
- 결과는 `/kaggle/working/` 최상위에 직접 저장하며, 마지막 셀에서 **다운로드용 ZIP과 파일 링크**를 출력합니다.
- 두 파일: `C_lightgcn_val_top100.csv.gz`, `C_lightgcn_final_top100.csv.gz` (각 사용자 100행, `user_id,item_id,score,rank`).

⚠️ `Run All`만 실행한 인터랙티브 노트북은 세션 종료 시 파일이 사라질 수 있습니다. 저장된 버전의 `Output`과 실행 중 세션의 `Files`는 다릅니다. `Save & Run All` 완료 후 **Versions → Output**에서 다운로드하세요.


In [ ]:
from pathlib import Path
import os, shutil, zipfile, subprocess, sys, time
from IPython.display import display, FileLink

WORK = Path('/kaggle/working')
ROOT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)
assert ROOT.exists(), 'Kaggle Input이 없습니다.'

# 연결된 Dataset에서 데이터 경로 자동 탐색
v = list(ROOT.rglob('amazon_topk_teamkit/validation/manifest.json'))
if not v:
    v = [p for p in ROOT.rglob('manifest.json') if (p.parent/'val_train.csv').exists() and (p.parent/'val_users.csv').exists()]
assert v, '공통 Validation이 없습니다: Add Input → swaidataset/amazon-topk-teamkit'
VAL = v[0].parent

tr = [p for p in ROOT.rglob('train.csv') if 'competitions' in str(p) and (p.parent/'test.csv').exists()]
if not tr:
    tr = [p for p in ROOT.rglob('train.csv') if (p.parent/'test.csv').exists() and 'amazon_topk_teamkit' not in str(p)]
assert tr, '대회 train.csv/test.csv가 없습니다: Competition Dataset을 Add Input으로 연결하세요.'
TRAIN = tr[0]
TEST = TRAIN.parent/'test.csv'

print('✅ Validation:', VAL)
print('✅ Original train:', TRAIN)
print('✅ Test users:', TEST)
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else '없음')


In [ ]:
%%writefile /kaggle/working/graph_models.py
"""Single-GPU LightGCN / NGCF experiments for the shared Amazon Top-20 split.

Example:
    python graph_models.py --model lightgcn \
      --fit ../amazon_topk_teamkit/validation/val_train.csv \
      --users ../amazon_topk_teamkit/validation/val_users.csv \
      --truth ../amazon_topk_teamkit/validation/val_truth.csv \
      --out lightgcn_val.csv

No metadata or reviews are read. Train interactions are the only graph edges.
"""

from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None


TOP_K = 20
_BaseModel = nn.Module if nn is not None else object


def _id_key(value: str):
    return (0, int(value)) if value.isdecimal() else (1, value)


def _csv_rows(path: str | Path, required: tuple[str, ...]):
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
            raise ValueError(f"{path} must contain {', '.join(required)}")
        for row in reader:
            if None in row:
                raise ValueError(f"Malformed CSV row in {path}")
            yield row


def _identifier(value: str | None, column: str) -> str:
    if not value or value != value.strip() or any(char.isspace() for char in value):
        raise ValueError(f"Invalid {column}: {value!r}")
    return value


@dataclass
class InteractionData:
    user_ids: list[str]
    item_ids: list[str]
    user_index: dict[str, int]
    item_index: dict[str, int]
    positive_users: list[int]
    positive_items: list[int]
    seen_items: dict[int, set[int]]
    target_users: list[str]

    @property
    def num_users(self) -> int:
        return len(self.user_ids)

    @property
    def num_items(self) -> int:
        return len(self.item_ids)

    @property
    def num_edges(self) -> int:
        return len(self.positive_users)


def load_data(fit_path: str | Path, users_path: str | Path) -> InteractionData:
    """Index unique positive edges; do not filter interactions by rating."""
    edges: set[tuple[str, str]] = set()
    for row in _csv_rows(fit_path, ("user_id", "item_id")):
        edges.add((_identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")))
    if not edges:
        raise ValueError("Empty fit interactions")

    target_users: list[str] = []
    target_set: set[str] = set()
    for row in _csv_rows(users_path, ("user_id",)):
        user_id = _identifier(row["user_id"], "user_id")
        if user_id in target_set:
            raise ValueError(f"Duplicate target user: {user_id}")
        target_set.add(user_id)
        target_users.append(user_id)

    train_users = {user for user, _ in edges}
    unseen_users = target_set - train_users
    if unseen_users:
        raise ValueError(f"{len(unseen_users)} target users have no training interaction")
    user_ids = sorted(train_users, key=_id_key)
    item_ids = sorted({item for _, item in edges}, key=_id_key)
    if len(item_ids) <= TOP_K:
        raise ValueError("Need more than 20 known items to recommend 20 unseen items")
    user_index = {value: index for index, value in enumerate(user_ids)}
    item_index = {value: index for index, value in enumerate(item_ids)}
    positive_users: list[int] = []
    positive_items: list[int] = []
    seen_items: dict[int, set[int]] = defaultdict(set)
    for user, item in sorted(edges, key=lambda pair: (_id_key(pair[0]), _id_key(pair[1]))):
        u, i = user_index[user], item_index[item]
        positive_users.append(u)
        positive_items.append(i)
        seen_items[u].add(i)
    for user in target_users:
        if len(seen_items[user_index[user]]) + TOP_K > len(item_ids):
            raise ValueError(f"User {user} has fewer than 20 unseen items")
    return InteractionData(
        user_ids, item_ids, user_index, item_index, positive_users,
        positive_items, dict(seen_items), target_users,
    )


def load_truth(path: str | Path, target_users: Sequence[str]) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    for row in _csv_rows(path, ("user_id", "item_id")):
        user, item = _identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")
        if item in truth[user]:
            raise ValueError(f"Duplicate truth pair: {user}, {item}")
        truth[user].add(item)
    if set(truth) != set(target_users):
        raise ValueError("Truth and target user sets differ")
    return dict(truth)


def recall_at_20(predictions: dict[str, list[str]], truth: dict[str, set[str]]) -> dict:
    """Pooled competition Recall@20: total hits / sum(min(20, |GT_u|))."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user]) for user, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if not denominator:
        raise ValueError("No truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def normalized_graph(data: InteractionData, device):
    """D^-1/2 A D^-1/2 for the undirected user-item graph."""
    if torch is None:
        raise RuntimeError("PyTorch is required to build the graph")
    u = torch.tensor(data.positive_users, dtype=torch.long)
    i = torch.tensor(data.positive_items, dtype=torch.long) + data.num_users
    rows = torch.cat((u, i))
    cols = torch.cat((i, u))
    degree = torch.bincount(rows, minlength=data.num_users + data.num_items).float()
    values = (degree[rows] * degree[cols]).rsqrt()
    indices = torch.stack((rows, cols)).to(device)
    return torch.sparse_coo_tensor(
        indices, values.to(device),
        (data.num_users + data.num_items, data.num_users + data.num_items),
        device=device,
    ).coalesce()


def sparse_edge_dropout(graph, probability: float):
    if probability <= 0:
        return graph
    if not 0 <= probability < 1:
        raise ValueError("Edge dropout must be in [0, 1)")
    keep = torch.rand(graph._nnz(), device=graph.device) >= probability
    return torch.sparse_coo_tensor(
        graph.indices()[:, keep], graph.values()[keep] / (1 - probability),
        graph.shape, device=graph.device,
    ).coalesce()


class LightGCN(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int):
        if nn is None:
            raise RuntimeError("PyTorch is required to train LightGCN")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.layers = layers
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(graph, current)
            layer_outputs.append(current)
        final = torch.stack(layer_outputs, dim=0).mean(dim=0)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


class NGCF(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int, message_dropout: float):
        if nn is None:
            raise RuntimeError("PyTorch is required to train NGCF")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.sum_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.bi_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.message_dropout = message_dropout
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        for layer in (*self.sum_linears, *self.bi_linears):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for sum_linear, bi_linear in zip(self.sum_linears, self.bi_linears):
            neighbors = torch.sparse.mm(graph, current)
            current = F.leaky_relu(sum_linear(current + neighbors) + bi_linear(current * neighbors), negative_slope=0.2)
            current = F.dropout(current, p=self.message_dropout, training=self.training)
            current = F.normalize(current, p=2, dim=1)
            layer_outputs.append(current)
        final = torch.cat(layer_outputs, dim=1)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


def sample_triples(data: InteractionData, count: int, rng: random.Random, device):
    """Sample one uniformly random unseen item per sampled positive edge."""
    if count <= 0:
        raise ValueError("samples-per-epoch must be positive")
    edge_indices = [rng.randrange(data.num_edges) for _ in range(count)]
    users = [data.positive_users[index] for index in edge_indices]
    positive = [data.positive_items[index] for index in edge_indices]
    negative = [rng.randrange(data.num_items) for _ in range(count)]
    for position, (user, item) in enumerate(zip(users, negative)):
        if item in data.seen_items[user]:
            while True:
                candidate = rng.randrange(data.num_items)
                if candidate not in data.seen_items[user]:
                    negative[position] = candidate
                    break
    return (
        torch.tensor(users, dtype=torch.long, device=device),
        torch.tensor(positive, dtype=torch.long, device=device),
        torch.tensor(negative, dtype=torch.long, device=device),
    )


@torch.no_grad() if torch is not None else (lambda function: function)
def predict_top20(model, graph, data: InteractionData, user_batch_size: int) -> dict[str, list[str]]:
    """Score all known items in bounded user batches and mask fit interactions."""
    if user_batch_size <= 0:
        raise ValueError("eval-user-batch-size must be positive")
    model.eval()
    user_vectors, item_vectors = model(graph)
    item_matrix = item_vectors.T.contiguous()
    predictions: dict[str, list[str]] = {}
    for start in range(0, len(data.target_users), user_batch_size):
        batch = data.target_users[start : start + user_batch_size]
        user_indices = torch.tensor([data.user_index[user] for user in batch], dtype=torch.long, device=graph.device)
        scores = user_vectors[user_indices] @ item_matrix
        masked_rows: list[int] = []
        masked_columns: list[int] = []
        for local_row, user_index in enumerate(user_indices.tolist()):
            for item_index in data.seen_items[user_index]:
                masked_rows.append(local_row)
                masked_columns.append(item_index)
        if masked_rows:
            row_index = torch.tensor(masked_rows, dtype=torch.long, device=graph.device)
            column_index = torch.tensor(masked_columns, dtype=torch.long, device=graph.device)
            scores[row_index, column_index] = -float("inf")
        top = torch.topk(scores, k=TOP_K, dim=1).indices.cpu().tolist()
        for user_id, item_indices in zip(batch, top):
            predictions[user_id] = [data.item_ids[index] for index in item_indices]
    return predictions


def write_predictions(path: str | Path, users: Sequence[str], predictions: dict[str, list[str]]) -> None:
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user in users:
            writer.writerow((user, " ".join(predictions[user])))


def _device(name: str):
    if torch is None:
        raise RuntimeError("PyTorch is not installed. Use Kaggle GPU Notebook or install torch.")
    if name == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if name.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    return torch.device(name)


def _training_loss(model, user_vectors, item_vectors, sampled, regularization: float):
    users, positives, negatives = sampled
    positive_scores = (user_vectors[users] * item_vectors[positives]).sum(dim=1)
    negative_scores = (user_vectors[users] * item_vectors[negatives]).sum(dim=1)
    bpr = F.softplus(negative_scores - positive_scores).mean()
    raw_u = model.user_embedding(users)
    raw_p = model.item_embedding(positives)
    raw_n = model.item_embedding(negatives)
    reg = (raw_u.square().sum(dim=1) + raw_p.square().sum(dim=1) + raw_n.square().sum(dim=1)).mean() / 2
    return bpr + regularization * reg, bpr, reg


def run(args: argparse.Namespace) -> dict:
    device = _device(args.device)
    if args.layers < 1 or args.embedding_dim < 1 or args.epochs < 1:
        raise ValueError("layers, embedding-dim and epochs must be positive")
    if not 0 <= args.edge_dropout < 1 or not 0 <= args.message_dropout < 1:
        raise ValueError("Dropout probabilities must be in [0, 1)")
    if args.reg < 0 or args.lr <= 0:
        raise ValueError("reg must be nonnegative and lr positive")
    torch.manual_seed(args.seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(args.seed)
    rng = random.Random(args.seed)
    data = load_data(args.fit, args.users)
    truth = load_truth(args.truth, data.target_users) if args.truth else None
    graph = normalized_graph(data, device)
    if args.model == "lightgcn":
        model = LightGCN(data.num_users, data.num_items, args.embedding_dim, args.layers)
    else:
        model = NGCF(data.num_users, data.num_items, args.embedding_dim, args.layers, args.message_dropout)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    print(json.dumps({
        "model": args.model, "device": str(device), "users": data.num_users,
        "items": data.num_items, "fit_edges": data.num_edges,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
    }), flush=True)

    best_score = -math.inf
    best_state = None
    best_epoch = None
    stale_evaluations = 0
    last_loss = None
    trained_epochs = 0
    for epoch in range(1, args.epochs + 1):
        trained_epochs = epoch
        start = time.monotonic()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, args.samples_per_epoch, rng, device)
        epoch_graph = sparse_edge_dropout(graph, args.edge_dropout) if args.model == "ngcf" else graph
        user_vectors, item_vectors = model(epoch_graph)
        loss, bpr, reg = _training_loss(model, user_vectors, item_vectors, sampled, args.reg)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        record = {
            "epoch": epoch, "loss": round(last_loss, 6),
            "bpr": round(float(bpr.detach().cpu()), 6),
            "regularization": round(float(reg.detach().cpu()), 6),
            "seconds": round(time.monotonic() - start, 2),
        }
        del sampled, user_vectors, item_vectors, loss, bpr, reg
        should_evaluate = truth is not None and (epoch == 1 or epoch % args.eval_every == 0 or epoch == args.epochs)
        if should_evaluate:
            predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
            metric = recall_at_20(predictions, truth)
            record.update(metric)
            if metric["recall_at_20"] > best_score:
                best_score = metric["recall_at_20"]
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                stale_evaluations = 0
                record["best"] = True
            else:
                stale_evaluations += 1
                record["best"] = False
        print(json.dumps(record), flush=True)
        if should_evaluate and args.patience > 0 and stale_evaluations >= args.patience:
            print(json.dumps({"early_stop_epoch": epoch}), flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
    write_predictions(args.out, data.target_users, predictions)
    result = {
        "model": args.model, "device": str(device), "fit_edges": data.num_edges,
        "users": len(data.target_users), "output": str(args.out), "last_loss": last_loss,
        "trained_epochs": trained_epochs, "best_epoch": best_epoch,
        "epochs_requested": args.epochs, "seed": args.seed,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
        "learning_rate": args.lr, "regularization": args.reg,
        "edge_dropout": args.edge_dropout if args.model == "ngcf" else 0.0,
        "message_dropout": args.message_dropout if args.model == "ngcf" else 0.0,
    }
    if truth is not None:
        result.update(recall_at_20(predictions, truth))
    print(json.dumps({"final": result}), flush=True)
    return result


def main(argv: Sequence[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description="LightGCN / NGCF on the shared Amazon Top-20 split")
    parser.add_argument("--model", choices=("lightgcn", "ngcf"), required=True)
    parser.add_argument("--fit", required=True, help="val_train.csv or full train.csv")
    parser.add_argument("--users", required=True, help="val_users.csv or test.csv")
    parser.add_argument("--truth", help="val_truth.csv; omit for final test predictions")
    parser.add_argument("--out", required=True, help="Prediction CSV")
    parser.add_argument("--metrics-out", help="Optional JSON file for run settings and results")
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--embedding-dim", type=int, default=64)
    parser.add_argument("--layers", type=int, default=3)
    parser.add_argument("--samples-per-epoch", type=int, default=100000)
    parser.add_argument("--eval-user-batch-size", type=int, default=256)
    parser.add_argument("--eval-every", type=int, default=5)
    parser.add_argument("--patience", type=int, default=4, help="Validation evaluations without improvement; 0 disables")
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--reg", type=float, default=0.0001)
    parser.add_argument("--edge-dropout", type=float, default=0.0, help="Optional NGCF edge dropout")
    parser.add_argument("--message-dropout", type=float, default=0.1, help="NGCF only")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto", help="auto, cpu, cuda:0")
    args = parser.parse_args(argv)
    if args.eval_every < 1 or args.patience < 0:
        parser.error("eval-every must be positive and patience nonnegative")
    result = run(args)
    if args.metrics_out:
        Path(args.metrics_out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/teamkit.py
"""Shared, standard-library-only evaluation utilities for the Amazon Top-20 team.

All interactions are targets, regardless of rating. IDs are kept as strings so
CSV round-trips do not change them. The public CSV contract is user_id,item_id.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import os
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Iterable, Mapping, Sequence


TOP_K = 20
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.20


@contextmanager
def open_source(path: str | None, competition_zip: str | None, member: str):
    """Open a UTF-8 file, either directly or as a named competition ZIP member."""
    if (path is None) == (competition_zip is None):
        raise ValueError("Provide exactly one of a file path or --competition-zip")
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive:
            with archive.open(member) as raw:
                with io.TextIOWrapper(raw, encoding="utf-8-sig", newline="") as stream:
                    yield stream
    else:
        with open(path, "r", encoding="utf-8-sig", newline="") as stream:
            yield stream


def _reader(stream, required: Sequence[str]) -> csv.DictReader:
    reader = csv.DictReader(stream)
    if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
        raise ValueError(f"Required columns: {', '.join(required)}")
    return reader


def _clean_id(value: str | None, name: str) -> str:
    if value is None or value == "" or value != value.strip() or any(c.isspace() for c in value):
        raise ValueError(f"Invalid {name}: {value!r}")
    return value


def _stable_rank(seed: int, user_id: str, item_id: str) -> bytes:
    payload = f"{seed}\0{user_id}\0{item_id}".encode("utf-8")
    return hashlib.blake2b(payload, digest_size=16).digest()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _source_sha256(path: str | None, competition_zip: str | None, member: str) -> str:
    digest = hashlib.sha256()
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive, archive.open(member) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    else:
        with open(path, "rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def make_split(
    train_path: str | None,
    competition_zip: str | None,
    output_dir: str | Path,
    seed: int = SPLIT_SEED,
    fraction: float = HOLDOUT_FRACTION,
    overwrite: bool = False,
) -> dict:
    """Make a stable, per-user split on unique user-item pairs.

    Hold out ceil(fraction * interactions), at most 20, leaving at least one
    interaction in fit. The hash rank is independent of input row order.
    """
    if not 0 < fraction < 1:
        raise ValueError("fraction must be between 0 and 1")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    names = ("val_train.csv", "val_truth.csv", "val_users.csv", "manifest.json")
    existing = [name for name in names if (output_dir / name).exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Existing split files: {existing}; use --overwrite to replace")

    with open_source(train_path, competition_zip, "train.csv") as stream:
        reader = _reader(stream, ("user_id", "item_id"))
        columns = list(reader.fieldnames or [])
        rows: list[tuple[str, ...]] = []
        user_items: dict[str, list[str]] = defaultdict(list)
        pairs: set[tuple[str, str]] = set()
        for row in reader:
            if None in row:
                raise ValueError("Malformed train.csv row with extra fields")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            pair = (user_id, item_id)
            if pair in pairs:
                raise ValueError(f"Duplicate interaction: {pair}")
            pairs.add(pair)
            user_items[user_id].append(item_id)
            rows.append(tuple(row[column] for column in columns))

    held: set[tuple[str, str]] = set()
    for user_id, items in user_items.items():
        if len(items) < 2:
            continue
        n_held = min(TOP_K, len(items) - 1, max(1, math.ceil(fraction * len(items))))
        chosen = sorted(items, key=lambda item: (_stable_rank(seed, user_id, item), item))[:n_held]
        held.update((user_id, item) for item in chosen)

    fit_path = output_dir / "val_train.csv"
    truth_path = output_dir / "val_truth.csv"
    users_path = output_dir / "val_users.csv"
    user_col, item_col = columns.index("user_id"), columns.index("item_id")
    with fit_path.open("w", encoding="utf-8", newline="") as fit_stream, truth_path.open(
        "w", encoding="utf-8", newline=""
    ) as truth_stream:
        fit_writer, truth_writer = csv.writer(fit_stream), csv.writer(truth_stream)
        fit_writer.writerow(columns)
        truth_writer.writerow(("user_id", "item_id"))
        for row in rows:
            pair = (row[user_col], row[item_col])
            if pair in held:
                truth_writer.writerow(pair)
            else:
                fit_writer.writerow(row)

    with users_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id",))
        for user_id in user_items:
            if len(user_items[user_id]) >= 2:
                writer.writerow((user_id,))

    source_hash = _source_sha256(train_path, competition_zip, "train.csv")
    split_key = f"{source_hash}|blake2b16|{seed}|{fraction}|ceil20keep1"
    manifest = {
        "split_id": hashlib.sha256(split_key.encode()).hexdigest()[:16],
        "algorithm": "per-user blake2b-128 rank over seed, user_id, item_id",
        "seed": seed,
        "holdout_fraction": fraction,
        "holdout_count": "min(20, n-1, max(1, ceil(fraction*n))) for n>=2",
        "source_train_sha256": source_hash,
        "source_rows": len(rows),
        "source_users": len(user_items),
        "fit_rows": len(rows) - len(held),
        "truth_rows": len(held),
        "validation_users": sum(len(items) >= 2 for items in user_items.values()),
        "files_sha256": {name: _sha256(output_dir / name) for name in names[:3]},
    }
    with (output_dir / "manifest.json").open("w", encoding="utf-8") as stream:
        json.dump(manifest, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    return manifest


def verify_split(output_dir: str | Path) -> dict:
    output_dir = Path(output_dir)
    with (output_dir / "manifest.json").open("r", encoding="utf-8") as stream:
        manifest = json.load(stream)
    for name, expected in manifest["files_sha256"].items():
        actual = _sha256(output_dir / name)
        if actual != expected:
            raise ValueError(f"Split file checksum mismatch: {name}")
    return manifest


def read_users(path: str | Path) -> list[str]:
    users: list[str] = []
    seen: set[str] = set()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id",)):
            if None in row:
                raise ValueError(f"Malformed user CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in seen:
                raise ValueError(f"Duplicate user in {path}: {user_id}")
            seen.add(user_id)
            users.append(user_id)
    return users


def read_interactions(path: str | Path) -> tuple[dict[str, set[str]], Counter[str]]:
    """Return seen items by user and item popularity from a fit CSV."""
    seen: dict[str, set[str]] = defaultdict(set)
    popularity: Counter[str] = Counter()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed interaction CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id not in seen[user_id]:
                seen[user_id].add(item_id)
                popularity[item_id] += 1
    return dict(seen), popularity


def read_truth(path: str | Path) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed truth CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id in truth[user_id]:
                raise ValueError(f"Duplicate truth item for user {user_id}: {item_id}")
            truth[user_id].add(item_id)
    return dict(truth)


def read_predictions(path: str | Path) -> dict[str, list[str]]:
    predictions: dict[str, list[str]] = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["user_id", "item_id"]:
            raise ValueError("Prediction columns must be exactly user_id,item_id")
        for row in reader:
            if None in row:
                raise ValueError("Malformed prediction CSV row")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in predictions:
                raise ValueError(f"Duplicate prediction user: {user_id}")
            value = row["item_id"]
            if value is None:
                raise ValueError(f"Missing item list for user {user_id}")
            items = value.split(" ")
            if len(items) != TOP_K or any(not item or item != item.strip() for item in items):
                raise ValueError(f"User {user_id} needs exactly {TOP_K} space-separated item IDs")
            if len(set(items)) != TOP_K:
                raise ValueError(f"Duplicate recommended item for user {user_id}")
            predictions[user_id] = items
    return predictions


def validate_predictions(
    predictions: Mapping[str, Sequence[str]],
    expected_users: Iterable[str],
    seen: Mapping[str, set[str]],
    catalog: set[str],
) -> None:
    expected = set(expected_users)
    actual = set(predictions)
    if actual != expected:
        missing, extra = expected - actual, actual - expected
        raise ValueError(f"User mismatch: {len(missing)} missing, {len(extra)} extra")
    for user_id, items in predictions.items():
        if len(items) != TOP_K or len(set(items)) != TOP_K:
            raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
        unknown = set(items) - catalog
        if unknown:
            raise ValueError(f"Unknown item for user {user_id}: {sorted(unknown)[:3]}")
        repeated = set(items) & seen.get(user_id, set())
        if repeated:
            raise ValueError(f"Seen item recommended to user {user_id}: {sorted(repeated)[:3]}")


def recall_at_20(predictions: Mapping[str, Sequence[str]], truth: Mapping[str, set[str]]) -> dict:
    """Competition-style pooled Recall@20, not mean of user-level recalls."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user_id]) for user_id, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if denominator == 0:
        raise ValueError("No validation truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def top20_unseen(
    user_id: str,
    ranked_candidates: Iterable[str],
    seen: Mapping[str, set[str]],
    fallback_items: Iterable[str],
    catalog: set[str],
) -> list[str]:
    """Filter a model ranking; fill short lists with popular unseen items."""
    chosen: list[str] = []
    used = set(seen.get(user_id, set()))
    for source in (ranked_candidates, fallback_items):
        for item_id in source:
            item_id = str(item_id)
            if item_id not in catalog or item_id in used:
                continue
            chosen.append(item_id)
            used.add(item_id)
            if len(chosen) == TOP_K:
                return chosen
    raise ValueError(f"Fewer than {TOP_K} available unseen items for user {user_id}")


def write_predictions(path: str | Path, users: Iterable[str], predictions: Mapping[str, Sequence[str]]) -> None:
    """Write one row per user in the provided order, using single spaces."""
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user_id in users:
            items = predictions[user_id]
            if len(items) != TOP_K or len(set(items)) != TOP_K:
                raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
            writer.writerow((user_id, " ".join(map(str, items))))


def filter_reviews(
    fit_path: str | Path,
    output_path: str | Path,
    reviews_path: str | None = None,
    competition_zip: str | None = None,
) -> dict:
    """Keep only training-pair reviews to prevent validation target leakage."""
    seen, _ = read_interactions(fit_path)
    allowed = {(user_id, item_id) for user_id, items in seen.items() for item_id in items}
    found: set[tuple[str, str]] = set()
    written = 0
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_path = tempfile.mkstemp(prefix=".reviews-", suffix=".tmp", dir=output_path.parent)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as output, open_source(
            reviews_path, competition_zip, "train_reviews.jsonl"
        ) as source:
            for line in source:
                review = json.loads(line)
                pair = (str(review["user_id"]), str(review["item_id"]))
                if pair in allowed:
                    output.write(line)
                    found.add(pair)
                    written += 1
        if found != allowed:
            raise ValueError(f"Review coverage mismatch: {len(allowed - found)} fit pairs have no review")
        os.replace(temporary_path, output_path)
    finally:
        if os.path.exists(temporary_path):
            os.unlink(temporary_path)
    return {"written_reviews": written, "distinct_fit_pairs": len(allowed)}


def _print_json(value: dict) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Shared Amazon Top-20 evaluation tools")
    commands = parser.add_subparsers(dest="command", required=True)

    split = commands.add_parser("split", help="Create deterministic validation CSVs")
    source = split.add_mutually_exclusive_group(required=True)
    source.add_argument("--train", help="Path to extracted train.csv")
    source.add_argument("--competition-zip", help="Path to original competition ZIP")
    split.add_argument("--out", required=True, help="Output directory")
    split.add_argument("--seed", type=int, default=SPLIT_SEED)
    split.add_argument("--fraction", type=float, default=HOLDOUT_FRACTION)
    split.add_argument("--overwrite", action="store_true")

    verify = commands.add_parser("verify-split", help="Check shared split file hashes")
    verify.add_argument("--dir", required=True)

    score = commands.add_parser("score", help="Validate and score validation predictions")
    score.add_argument("--pred", required=True)
    score.add_argument("--fit", required=True)
    score.add_argument("--truth", required=True)

    check = commands.add_parser("check", help="Validate a final test submission")
    check.add_argument("--pred", required=True)
    check.add_argument("--train", required=True)
    check.add_argument("--test", required=True)

    reviews = commands.add_parser("filter-reviews", help="Remove held-out reviews")
    reviews.add_argument("--fit", required=True)
    review_source = reviews.add_mutually_exclusive_group(required=True)
    review_source.add_argument("--reviews", help="Path to extracted train_reviews.jsonl")
    review_source.add_argument("--competition-zip", help="Path to original competition ZIP")
    reviews.add_argument("--out", required=True)

    args = parser.parse_args(argv)
    if args.command == "split":
        _print_json(make_split(args.train, args.competition_zip, args.out, args.seed, args.fraction, args.overwrite))
    elif args.command == "verify-split":
        manifest = verify_split(args.dir)
        _print_json({"split_id": manifest["split_id"], "verified": True})
    elif args.command == "score":
        seen, popularity = read_interactions(args.fit)
        truth = read_truth(args.truth)
        predictions = read_predictions(args.pred)
        catalog = set(popularity) | {item for items in truth.values() for item in items}
        validate_predictions(predictions, truth, seen, catalog)
        result = recall_at_20(predictions, truth)
        result["valid_submission_format"] = True
        _print_json(result)
    elif args.command == "check":
        seen, popularity = read_interactions(args.train)
        users = read_users(args.test)
        predictions = read_predictions(args.pred)
        validate_predictions(predictions, users, seen, set(popularity))
        _print_json({"valid": True, "users": len(users), "items_per_user": TOP_K})
    elif args.command == "filter-reviews":
        _print_json(filter_reviews(args.fit, args.out, args.reviews, args.competition_zip))
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except (ValueError, FileExistsError, FileNotFoundError, KeyError, zipfile.BadZipFile) as error:
        print(f"ERROR: {error}", file=sys.stderr)
        sys.exit(1)


In [ ]:
%%writefile /kaggle/working/lightgcn_top100_export.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Generate separate validation/final scored Top-100 files with the C group's LightGCN.

This script uses the *original* graph_models.py LightGCN, BPR loss, edge sampler,
normalized graph, and data ID handling (no substitute algorithm). The original
predict_top20 routine writes only 20 IDs without scores; this exporter streams
100 **scored** recommendations per target user instead.

Run (Kaggle working directory, with graph_models.py in the same directory):
  python lightgcn_top100_export.py \
    --validation-dir /kaggle/input/datasets/swaidataset/amazon-topk-teamkit/amazon_topk_teamkit/validation \
    --full-train /kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train.csv \
    --test-users /kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/test.csv \
    --output-dir /kaggle/working/C_lightgcn_top100

Optional: --seeds 42,2026,3407 combines multiple seeds' **scores**. Single
seed 42 is the default original-best single-run configuration, not a measured
multi-seed result.
"""
from __future__ import annotations

import argparse
import copy
import csv
import gzip
import json
import math
import random
import sys
from pathlib import Path

import torch
from torch.nn import functional as F

try:
    import graph_models as gm
except ImportError as exc:
    raise SystemExit('Missing graph_models.py. Put the original C model implementation beside this script.') from exc

SPLIT_ID = 'f0dfad2f4431a876'
TOPK = 100


def load_truth(path: Path) -> dict[str, set[str]]:
    return {u: s for u, s in gm.load_truth(path, list(read_users(path))).items()}


def read_users(path: Path) -> list[str]:
    # For val_truth.csv, we use unique user IDs; for val_users/test.csv, one row/user is expected.
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        rdr = csv.DictReader(f)
        if not rdr.fieldnames or 'user_id' not in rdr.fieldnames:
            raise ValueError(f'{path} needs user_id column')
        return list(dict.fromkeys(row['user_id'] for row in rdr))


def check_validation_split(validation: Path):
    manifest = validation / 'manifest.json'
    if not manifest.exists():
        raise FileNotFoundError(f'Missing validation manifest: {manifest}')
    data = json.loads(manifest.read_text(encoding='utf8'))
    if data.get('split_id') != SPLIT_ID:
        raise ValueError(f'Unexpected validation split: {data.get("split_id")}')
    try:
        import teamkit
        result = teamkit.verify_split(validation)
        if result.get('split_id') != SPLIT_ID:
            raise ValueError('teamkit.verify_split split ID mismatch')
    except ImportError:
        print('WARNING: teamkit.py unavailable: checked manifest split ID only, not dataset hashes.', flush=True)
    print('Validation split:', SPLIT_ID, flush=True)


def train_one(data: gm.InteractionData, graph: torch.Tensor, device: torch.device,
              *, seed: int, epochs: int, dim: int, layers: int, lr: float,
              reg: float, samples_per_epoch: int, truth_file: Path | None,
              eval_every: int, eval_batch: int):
    torch.manual_seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
    rng = random.Random(seed)
    model = gm.LightGCN(data.num_users, data.num_items, dim, layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_epoch, best_state = float('-inf'), 0, None
    truth = gm.load_truth(truth_file, data.target_users) if truth_file else None

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = gm.sample_triples(data, samples_per_epoch, rng, device)
        uv, iv = model(graph)
        loss, bpr, regularization = gm._training_loss(model, uv, iv, sampled, reg)
        loss.backward()
        optimizer.step()
        if epoch == 1 or epoch % 25 == 0 or epoch == epochs:
            print(json.dumps({'seed':seed, 'epoch':epoch, 'epochs':epochs,
                'loss':round(float(loss.detach().cpu()),6)}), flush=True)
        del sampled, uv, iv, loss, bpr, regularization
        if truth is not None and (epoch % eval_every == 0 or epoch == epochs):
            pred = gm.predict_top20(model, graph, data, eval_batch)
            result = gm.recall_at_20(pred, truth)
            metric = result['recall_at_20']
            if metric > best:
                best, best_epoch, best_state = metric, epoch, copy.deepcopy(model.state_dict())
            print(json.dumps({'seed':seed,'epoch':epoch, 'val_recall_at_20':metric,
                              'best_val_recall':best, 'best_epoch':best_epoch}),flush=True)
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        uv, iv = model(graph)
    print(json.dumps({'seed':seed,'best_epoch':best_epoch or epochs,
                      'best_recall':best if truth else None}),flush=True)
    return uv.detach().cpu(), iv.detach().cpu(), {'seed':seed, 'best_epoch':best_epoch or epochs,
                                                    'best_val_recall':best if truth else None}


def predict_and_write(data: gm.InteractionData, embeddings: list[tuple[torch.Tensor,torch.Tensor]],
                      out: Path, device: torch.device, batch_size: int, topk:int = TOPK):
    # Use raw dot product scores, average multiple seeds, and mask every seen item.
    if data.num_items < topk+1:
        raise ValueError('Need at least 101 catalog items for Top100')
    out.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(out, 'wt', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['user_id','item_id','score','rank'])
        n = len(data.target_users)
        for start in range(0,n,batch_size):
            users = data.target_users[start:start+batch_size]
            idx = torch.tensor([data.user_index[u] for u in users],device=device,dtype=torch.long)
            scores = None
            for uv,iv in embeddings:
                uv_batch = uv.to(device)[idx] if uv.device == device else uv[idx.cpu()].to(device)
                iv_gpu = iv.to(device)
                cur = uv_batch @ iv_gpu.t()
                scores = cur if scores is None else scores + cur
            scores /= len(embeddings)
            for ridx,user in enumerate(users):
                seen = list(data.seen_items[data.user_index[user]])
                scores[ridx,seen] = -torch.inf
            values, indices = torch.topk(scores,topk,dim=1)
            val_rows = values.cpu().tolist()
            ind_rows = indices.cpu().tolist()
            for user,vals,inds in zip(users,val_rows,ind_rows):
                if not all(math.isfinite(v) for v in vals):
                    raise ValueError(f'User {user} has <{topk} unseen items')
                for rank,(val,item_idx) in enumerate(zip(vals,inds),1):
                    writer.writerow([user,data.item_ids[item_idx],format(val,'.9g'),rank])
            if (start//batch_size)%100 == 0:
                print(f'{out.name}: {min(start+batch_size,n):,}/{n:,} users',flush=True)
    return out


def check_output(path:Path, users:list[str], topk:int=TOPK):
    # Streaming validation: no huge DataFrame is created.
    rows=0; last=None; rank=0; user_count=0; seen_items=set(); prev_score=float('inf')
    with gzip.open(path,'rt',encoding='utf8',newline='') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames != ['user_id','item_id','score','rank']:
            raise ValueError('Bad output header')
        for row in reader:
            u=row['user_id']; item=row['item_id']; value=float(row['score']); r=int(row['rank'])
            if u!=last:
                if last is not None and rank!=topk:
                    raise ValueError(f'User {last} has only {rank} items')
                if user_count >= len(users) or u != users[user_count]:
                    raise ValueError('User order mismatch / extra user')
                last=u;rank=0;seen_items=set();prev_score=float('inf');user_count+=1
            rank+=1
            if r!=rank or item in seen_items or value>prev_score+1e-6:
                raise ValueError(f'Invalid Top100 rank/duplicates/order for {u}')
            prev_score=value;seen_items.add(item);rows+=1
    if user_count!=len(users) or rank!=topk or rows!=len(users)*topk:
        raise ValueError('Output row count mismatch')
    print(f'✅ {path.name}: {rows:,} rows; {user_count:,} users, exactly Top{topk}',flush=True)


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--validation-dir',type=Path,required=True,
                    help='Folder with val_train.csv, val_users.csv, val_truth.csv, manifest.json')
    ap.add_argument('--full-train',type=Path,required=True)
    ap.add_argument('--test-users',type=Path,required=True)
    ap.add_argument('--output-dir',type=Path,default=Path('/kaggle/working/C_lightgcn_top100'))
    ap.add_argument('--epochs',type=int,default=575)
    ap.add_argument('--full-epochs',type=int,default=575,
                    help='Train full data for this many epochs (not using validation truth)')
    ap.add_argument('--dim',type=int,default=128)
    ap.add_argument('--layers',type=int,default=3)
    ap.add_argument('--lr',type=float,default=0.003)
    ap.add_argument('--reg',type=float,default=0.0001)
    ap.add_argument('--samples-per-epoch',type=int,default=100000)
    ap.add_argument('--seeds',default='42',help='Comma-separated integer seeds, e.g. 42,2026,3407')
    ap.add_argument('--eval-every',type=int,default=25)
    ap.add_argument('--batch-size',type=int,default=128)
    ap.add_argument('--device',default='auto')
    ap.add_argument('--skip-val',action='store_true',help='Only generate final file')
    ap.add_argument('--skip-final',action='store_true',help='Only generate validation file')
    args=ap.parse_args()
    if args.skip_val and args.skip_final:ap.error('Nothing to do')
    if args.epochs<1 or args.full_epochs<1 or args.samples_per_epoch<1 or args.batch_size<1:
        ap.error('epochs, full-epochs, samples-per-epoch, batch-size must be positive')
    seeds=[int(x) for x in args.seeds.split(',') if x.strip()]
    if not seeds or len(seeds)!=len(set(seeds)):ap.error('Provide one or more distinct seeds')
    device=torch.device(('cuda:0' if torch.cuda.is_available() else 'cpu') if args.device=='auto' else args.device)
    args.output_dir.mkdir(parents=True,exist_ok=True)
    check_validation_split(args.validation_dir)
    metadata={'model':'LightGCN','default_best_single_config':{'dim':128,'layers':3,'lr':0.003,'epochs':575},
              'config':{'dim':args.dim,'layers':args.layers,'lr':args.lr,'reg':args.reg,
                        'epochs':args.epochs,'full_epochs':args.full_epochs,'samples_per_epoch':args.samples_per_epoch,
                        'seeds':seeds,'topk':TOPK},'validation_split':SPLIT_ID,'results':{}}
    for stage in ('val','final'):
        if (stage=='val' and args.skip_val) or (stage=='final' and args.skip_final): continue
        fit=args.validation_dir/'val_train.csv' if stage=='val' else args.full_train
        users=args.validation_dir/'val_users.csv' if stage=='val' else args.test_users
        truth=args.validation_dir/'val_truth.csv' if stage=='val' else None
        epochs=args.epochs if stage=='val' else args.full_epochs
        data=gm.load_data(fit,users)
        for user in data.target_users:
            if data.num_items-len(data.seen_items[data.user_index[user]])<TOPK:
                raise ValueError(f'Not enough unseen items for user {user}')
        graph=gm.normalized_graph(data,device)
        all_embeddings=[];stats=[]
        for seed in seeds:
            uv,iv,meta=train_one(data,graph,device,seed=seed,epochs=epochs,dim=args.dim,
                   layers=args.layers,lr=args.lr,reg=args.reg,samples_per_epoch=args.samples_per_epoch,
                   truth_file=truth,eval_every=args.eval_every,eval_batch=args.batch_size)
            all_embeddings.append((uv,iv));stats.append(meta)
        path=args.output_dir/f'C_lightgcn_{stage}_top100.csv.gz'
        predict_and_write(data,all_embeddings,path,device,args.batch_size)
        check_output(path,data.target_users)
        metadata['results'][stage]={'path':str(path),'users':len(data.target_users),'rows':len(data.target_users)*TOPK,
                                    'trained_seeds':stats}
        (args.output_dir/'run_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf8')
        del graph,data,all_embeddings
    print('✅ All requested Top100 files generated successfully.',flush=True)

if __name__=='__main__':main()


## 기존 결과 복구 + 필요한 단계만 학습

먼저 같은 인터랙티브 세션의 이전 출력 폴더를 검색합니다. **둘 다 발견되면 학습하지 않습니다.** 새 세션이나 Save & Run All에서는 파일이 없을 수 있으므로 새로 학습합니다.


In [ ]:
import gzip
OUTPUTS = {
    'val': WORK/'C_lightgcn_val_top100.csv.gz',
    'final': WORK/'C_lightgcn_final_top100.csv.gz',
}

# 이미 생성된 출력이 하위 폴더에 있다면 /kaggle/working 최상위로 복사
for stage, dest in OUTPUTS.items():
    if not dest.exists():
        found = [p for p in WORK.rglob(dest.name) if p.is_file() and p != dest]
        if found:
            source=max(found, key=lambda p:p.stat().st_size)
            shutil.copy2(source, dest)
            print('✅ 기존 파일 복구:', source, '→', dest)

# 간단한 헤더/읽기 검증
for stage, dest in OUTPUTS.items():
    if dest.exists() and dest.stat().st_size>0:
        try:
            with gzip.open(dest,'rt',encoding='utf8') as f:
                assert f.readline().strip() == 'user_id,item_id,score,rank'
            print('✅ 재사용 가능:', dest.name, f'{dest.stat().st_size/1024**2:.2f} MB')
        except Exception:
            print('⚠️ 손상된 출력 재생성:', dest.name)
            dest.unlink()

need_val=not OUTPUTS['val'].exists()
need_final=not OUTPUTS['final'].exists()
print('학습 필요:', {'validation':need_val,'final':need_final})


In [ ]:
# 실험 당시 최고 LightGCN 단일 설정
SEEDS='42'
EPOCHS=575
FULL_EPOCHS=575

if need_val or need_final:
    cmd=[sys.executable, '-u', str(WORK/'lightgcn_top100_export.py'),
       '--validation-dir', str(VAL),
       '--full-train', str(TRAIN),
       '--test-users', str(TEST),
       '--output-dir', str(WORK),  # ★ 최상위 /kaggle/working에 직접 생성
       '--device', 'cuda:0' if __import__('torch').cuda.is_available() else 'cpu',
       '--dim','128', '--layers','3','--lr','0.003',
       '--epochs',str(EPOCHS),'--full-epochs',str(FULL_EPOCHS),
       '--seeds', SEEDS]
    if not need_val: cmd.append('--skip-val')
    if not need_final: cmd.append('--skip-final')
    print('학습 명령:', ' '.join(cmd), flush=True)
    logpath=WORK/'C_lightgcn_top100_training.log'
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    with logpath.open('w',encoding='utf8') as log:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env,cwd=WORK)
        for line in p.stdout:
            print(line,end='',flush=True)
            log.write(line);log.flush()
        rc=p.wait()
    if rc!=0: raise RuntimeError(f'LightGCN 실행 실패 exit={rc} · 로그 확인: {logpath}')
else:
    print('✅ 기존 Val/Final 두 파일을 찾았으므로 575 Epoch 재학습 생략')

# 실질적 파일 존재 확인 (로그만으로 성공 처리하지 않음)
for name,p in OUTPUTS.items():
    assert p.is_file() and p.stat().st_size>0, f'결과 파일 없음: {p}'
print('✅ 실제 파일 존재 검사 통과')


In [ ]:
import csv, gzip, zipfile

ZIP=WORK/'C_lightgcn_Top100_DOWNLOAD.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_STORED) as archive:
    for stage,path in OUTPUTS.items():
        assert path.exists() and path.stat().st_size>0, f'파일 없음: {path}'
        with gzip.open(path,'rt',encoding='utf8') as f:
            header=next(csv.reader(f))
        assert header == ['user_id','item_id','score','rank'], f'헤더 불일치: {header}'
        archive.write(path,arcname=path.name)

with zipfile.ZipFile(ZIP) as archive:
    assert len(archive.namelist())==2
    print('✅ ZIP 내부 파일 목록:', archive.namelist())

print('\n=== 실제 Kaggle 파일 위치 ===')
for stage,path in OUTPUTS.items():
    print(f'✅ {path}  ({path.stat().st_size/1024**2:.2f} MB)')
    display(FileLink(str(path)))
print(f'✅ 다운로드용 ZIP: {ZIP} ({ZIP.stat().st_size/1024**2:.2f} MB)')
display(FileLink(str(ZIP)))
print('\n⚠️ 주의: 위 링크는 현재 실행 세션용입니다.\nKaggle 저장된 버전의 Output에 영구 보존하려면 처음부터 Save Version → Save & Run All을 실행하고, 완료 후 Versions → Output에서 다운로드하세요.')
